In [1]:
import os
from dotenv import load_dotenv
load_dotenv(".env")

MODEL_ID = "claude-sonnet-4-6"
assert os.environ["ANTHROPIC_API_KEY"][:2] == "sk",\
       "Please specify the ANTHROPIC_API_KEY access token in keys.env file"

In [2]:
import nest_asyncio
nest_asyncio.apply()

In [3]:
from dataclasses import dataclass

@dataclass
class InventoryItem:
    name: str
    quantity_on_hand: int
    weekly_quantity_sold_past_n_weeks: [int]
    weeks_to_deliver: int

@dataclass
class Reorder:
    name: str
    quantity_to_order: int
    reason_to_reorder: str

items = [
    InventoryItem("itemA", 300, [50, 70, 80, 100], 2),
    InventoryItem("itemB", 100, [70, 80, 90, 70], 2),
    InventoryItem("itemC", 200, [80, 70, 90, 80], 1)
]

In [8]:
from pydantic_ai import Agent

agent = Agent(
    f"anthropic:{MODEL_ID}",
    system_prompt="You are an inventory manager who orders just in time.",
    output_type=list[Reorder]
)

result = agent.run_sync(f"""
Identify which of these items need to be reordered this week.

**Items**
{items}
""")

print(result.output)

[Reorder(name='itemB', quantity_to_order=60, reason_to_reorder="ItemB has 100 units on hand with an average weekly sales rate of 77.5 units/week and a 2-week delivery lead time. It needs ~155 units to cover the delivery period, but only has 100 — a shortfall of ~55 units. Reordering ~60 units this week ensures stock doesn't run out before the next delivery arrives."), Reorder(name='itemC', quantity_to_order=80, reason_to_reorder='ItemC has 200 units on hand with an average weekly sales rate of 80 units/week and a 1-week delivery lead time. It needs ~80 units to cover the delivery period, leaving only ~120 units as a buffer. Since we are just-in-time managing, ordering ~80 units this week replenishes exactly what will be sold before delivery arrives, keeping stock healthy without over-ordering.')]
